<div class="alert alert-block alert-info">
    
## Imports & Parameters
</div>

In [1]:
## Import
import os
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
import dask.dataframe as dd
from datetime import datetime, timedelta

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [2]:
# Calculate the start and end dates directly
start_date = datetime.today() - timedelta(days=120)
end_date = datetime.today() - timedelta(days=1)

print('Date range : {} to {}'.format(start_date.strftime('%Y%m%d'), end_date.strftime('%Y%m%d')))

# Generate the list of dates with a 15-day interval
date_list = [start_date.strftime('%Y%m%d')]
current_date = start_date

while current_date <= end_date:
    current_date += timedelta(days=15)  # Increment by 15 days
    if current_date <= end_date:
        date_list.append(current_date.strftime('%Y%m%d'))

print(date_list)

Date range : 20241014 to 20250210
['20241014', '20241029', '20241113', '20241128', '20241213', '20241228', '20250112', '20250127']


In [3]:
# Get the current working directory
cwd = os.getcwd()
local_extract = '/Users/E2074/local-datasets/customer/age-prediction/local_extract/'

print(cwd)
print(local_extract)

/Users/E2074/analytics/customer/age-prediction
/Users/E2074/local-datasets/customer/age-prediction/local_extract/


<div class="alert alert-block alert-info">
    
## Datasets
</div>

<div class="alert alert-block alert-warning">

### Metric Data
</div>

In [4]:
def get_metric_data():
    print('Query 1 running..')
    start_time = time.time()
    
    active_customer_base = f"""
    
        WITH funnel_data AS (
            
            SELECT
                customerid,
                SUM(ao_sessions_unique_daily) AS ao_sessions_unique_daily,
                SUM(fe_sessions_unique_daily) AS fe_sessions_unique_daily,
                SUM(rr_sessions_unique_daily) AS rr_sessions_unique_daily,
                SUM(gross_rides_daily) AS gross_rides_daily,
                SUM(net_rides_daily) AS net_rides_daily,
                SUM(active_day) AS active_day
            FROM
                (
                SELECT 
                    day,
                    customerid,
                    MAX(coalesce(gross_rides_daily, ao_sessions_unique_daily)) ao_sessions_unique_daily,
                    MAX(coalesce(gross_rides_daily, fe_sessions_unique_daily)) fe_sessions_unique_daily,
                    MAX(coalesce(gross_rides_daily, rr_sessions_unique_daily)) rr_sessions_unique_daily,
                    SUM(gross_rides_daily) gross_rides_daily,
                    SUM(net_rides_daily) net_rides_daily,
                    MAX(active_day) active_day
                FROM 
                    datasets.customer_rf_daily_kpi 
                WHERE 
                    day >= date_format(current_date - interval '120' day, '%Y-%m-%d')
                    AND day <= date_format(current_date, '%Y-%m-%d')
                    AND customerid NOT IN (SELECT customer_id FROM experiments_internal.customer_predicated_age_immutable WHERE confidence_tag != 'VERY_HIGH')
                    -- AND customerid = '6288c6694281e53bbe33d133' 
                GROUP BY 1,2
                )
            GROUP BY 1
            )
                
            SELECT * FROM funnel_data
            
    """
    
    df_active_customer_base = pd.read_sql(active_customer_base, connection)
    end_time = time.time()
    execution_time = round(((end_time - start_time)*1.00/60.00), 2)
    print("Execution time:", execution_time, "mins")
    df_active_customer_base.head(1)
    df_active_customer_base.to_parquet(local_extract + 'raw_customer_funnel.parquet', index=False)

In [5]:
# get_metric_data()

<div class="alert alert-block alert-warning">

### Appography, Demographic and Segment Data
</div>

In [6]:
def get_customer_demo_data():
    for date in date_list:
        print('Query running for ', date, ' ....')
        start_time = time.time()
        
        customer_demo_data = f"""
            WITH appography AS ( 
            
                SELECT
                    userid,
                    apps_installed
                FROM
                    datasets.customer_apps_installed_snapshot
                WHERE
                    DATE_FORMAT(from_unixtime(updated_epoch / 1000), '%Y%m%d') BETWEEN '{date}' AND
                    DATE_FORMAT(DATE_PARSE('{date}','%Y%m%d') + interval '4' day , '%Y%m%d')
                    -- BETWEEN date_format(current_date - interval '120' day, '%Y%m%d') AND date_format(current_date, '%Y%m%d')
                    AND userid NOT IN (SELECT customer_id FROM experiments_internal.customer_predicated_age_immutable WHERE confidence_tag != 'VERY_HIGH')
                    -- AND userid = '6288c6694281e53bbe33d133' 
                ),
            
                category AS (
                
                SELECT 
                    userid,
                    ai,
                    cameraFilter,
                    courier,
                    dating,
                    deliveryFood,
                    deliveryGrocery,
                    devotional,
                    driverApp,
                    ecommerce,
                    educational,
                    entertainment,
                    fantasySports,
                    financeInvestment,
                    financeNews,
                    financeTransactions,
                    gaming,
                    government,
                    health,
                    homeSecurity,
                    insurance,
                    jobSearch,
                    news,
                    office,
                    ott,
                    social,
                    streamingMusic,
                    telecom,
                    tools,
                    travelBookings,
                    travelMetro,
                    travelRidehailing,
                    vehicle,
                    wearable
                FROM 
                    reports.sql_ingestion_customer_appography_category_view
                WHERE
                    yyyymmdd = '20250126'
                    AND userid NOT IN (SELECT customer_id FROM experiments_internal.customer_predicated_age_immutable WHERE confidence_tag != 'VERY_HIGH')
                ),
                
                demographic_data AS ( 
            
                SELECT
                    customerId customer_id,
                    CASE WHEN age BETWEEN 15 AND 60 THEN age ELSE 0 END age,
                    COALESCE(gender, 'UNKNOWN') gender,
                    preferredTimeBucket
                FROM 
                    datasets.customer_single_view
                WHERE 
                    customerId NOT IN (SELECT customer_id FROM experiments_internal.customer_predicated_age_immutable WHERE confidence_tag != 'VERY_HIGH')
                ),
                
                segment AS (
                
                SELECT
                    customer_id,
                    taxi_ltr_segment ltr_segment,
                    taxi_retention_segments retention_segments,
                    taxi_regularity_segment regularity_segment,
                    taxi_service_affinity service_affinity
                    
                FROM 
                    datasets.iallocator_customer_segments
                WHERE 
                    run_date =  date_format(current_date - interval '2' day, '%Y-%m-%d')
                    AND taxi_recency_segment != 'INACTIVE' 
                )
                
                SELECT
                    appography.userid,
                    demographic_data.age,
                    demographic_data.gender,
                    demographic_data.preferredTimeBucket,
                    appography.apps_installed,
                    coalesce(ai, 0) ai,
                    coalesce(cameraFilter, 0) cameraFilter,
                    coalesce(courier, 0) courier,
                    coalesce(dating, 0) dating,
                    coalesce(deliveryFood, 0) deliveryFood,
                    coalesce(deliveryGrocery, 0) deliveryGrocery,
                    coalesce(devotional, 0) devotional,
                    coalesce(driverApp, 0) driverApp,
                    coalesce(ecommerce, 0) ecommerce,
                    coalesce(educational, 0) educational,
                    coalesce(entertainment, 0) entertainment,
                    coalesce(fantasySports, 0) fantasySports,
                    coalesce(financeInvestment, 0) financeInvestment,
                    coalesce(financeNews, 0) financeNews,
                    coalesce(financeTransactions, 0) financeTransactions,
                    coalesce(gaming, 0) gaming,
                    coalesce(government, 0) government,
                    coalesce(health, 0) health,
                    coalesce(homeSecurity, 0) homeSecurity,
                    coalesce(insurance, 0) insurance,
                    coalesce(jobSearch, 0) jobSearch,
                    coalesce(news, 0) news,
                    coalesce(office, 0) office,
                    coalesce(ott, 0) ott,
                    coalesce(social, 0) social,
                    coalesce(streamingMusic, 0) streamingMusic,
                    coalesce(telecom, 0) telecom,
                    coalesce(tools, 0) tools,
                    coalesce(travelBookings, 0) travelBookings,
                    coalesce(travelMetro, 0) travelMetro,
                    coalesce(travelRidehailing, 0) travelRidehailing,
                    coalesce(vehicle, 0) vehicle,
                    coalesce(wearable, 0) wearable,
                    coalesce(ltr_segment, 'UNKNOWN') ltr_segment,
                    coalesce(retention_segments, 'UNKNOWN') retention_segments,
                    coalesce(regularity_segment, 'UNKNOWN') regularity_segment,
                    coalesce(service_affinity, 'UNKNOWN') service_affinity
                FROM 
                    appography
                LEFT JOIN
                    category
                    ON appography.userid = category.userid
                LEFT JOIN 
                    demographic_data
                    ON appography.userid = demographic_data.customer_id
                LEFT JOIN 
                    segment
                    ON appography.userid = segment.customer_id
        """
        
        df_customer_demo_data = pd.read_sql(customer_demo_data, connection)
        end_time = time.time()
        execution_time = round(((end_time - start_time)*1.00/60.00), 2)
        print("Execution time:", execution_time, "mins")
        df_customer_demo_data.to_parquet(local_extract + 'demo/{}_customer_demo_data.parquet'.format(date), index=False)

In [7]:
# get_customer_demo_data()

<div class="alert alert-block alert-warning">

### Read Local Extract & Merge
</div>

#### Local Extract

In [8]:
def get_local_extract():

    start_time = time.time()
    # print('customer funnel')
    # df_active_customer_base = pd.read_parquet(local_extract + 'raw_customer_funnel.parquet')
    # df_active_customer_base.rename(columns={'customerid': 'customer_id'}, inplace=True)
    
    print('customer demo')
    
    # df_customer_demo_data = pd.read_parquet(local_extract + 'demo/dec_jan_customer_demo_data.parquet')
    df_customer_demo_data = pd.read_parquet(local_extract + 'demo/feb_customer_demo_data.parquet')
    # df_customer_demo_data = pd.read_parquet(local_extract + 'demo/oct_nov_customer_demo_data.parquet')
    
    df_customer_demo_data.rename(columns={'userid': 'customer_id'}, inplace=True)
    
    end_time = time.time()
    execution_time = round(((end_time - start_time)*1.00/60.00), 2)
    print("Execution time:", execution_time, "mins")
    
    return df_customer_demo_data #, df_active_customer_base

In [9]:
df_customer_demo_data = get_local_extract()

customer demo
Execution time: 0.49 mins


In [10]:
df_customer_demo_data.shape

(8213873, 42)

In [11]:
df_customer_demo_data.customer_id.nunique()
# 1092918

8213873

In [12]:
8213873/1700000

4.83169

In [13]:
# dec-jan 21858352 - 13
# feb 8213873 - 5
# oct-nov 9802560 - 6

In [14]:
for col in df_customer_demo_data.select_dtypes(include=['int64']).columns:
    df_customer_demo_data[col] = df_customer_demo_data[col].astype('int32')

df_customer_demo_data['chunk'] = df_customer_demo_data.index % 5  # Assign chunk numbers (0-4)
dfs = [group.reset_index(drop=True) for _, group in df_customer_demo_data.groupby('chunk')]

print([len(d) for d in dfs])  # Check row counts

[1642775, 1642775, 1642775, 1642774, 1642774]


In [15]:
dfs[0].head(5)

,customer_id,age,gender,preferredTimeBucket,apps_installed,ai,cameraFilter,courier,dating,deliveryFood,deliveryGrocery,devotional,driverApp,ecommerce,educational,entertainment,fantasySports,financeInvestment,financeNews,financeTransactions,gaming,government,health,homeSecurity,insurance,jobSearch,news,office,ott,social,streamingMusic,telecom,tools,travelBookings,travelMetro,travelRidehailing,vehicle,wearable,ltr_segment,retention_segments,regularity_segment,service_affinity,chunk
0,64c78f2819f4587c2edfc31c,0.0,MALE,Evening,"amazon alexa,axis mobile,cred,flipkart,gaana,g...",0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,1,1,0,0,1,0,1,1,1,1,1,0,1,1,0,1,1,0,PHH,SILVER,MONTHLY,UNKNOWN,0
1,622ef7334b3aa3b115549d61,0.0,MALE,Afternoon,"google pay,gmail,youtube,instagram,ludo king,m...",0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,1,1,0,1,1,0,1,0,0,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,0
2,6639c37dea110570cf5c0e72,28.0,MALE,Evening,"wynk music,cricbuzz,facebook,messenger,flipkar...",0,0,0,0,0,1,0,1,1,0,0,1,1,0,1,0,0,1,0,0,0,1,0,1,1,1,0,1,0,0,1,0,0,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,0
3,67a0829f64166bc6f2db2070,0.0,MALE,Afternoon,"gmail,youtube,ludo king,whatsapp,irctc rail co...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,HH,HH,UNKNOWN,UNKNOWN,0
4,677e920903e7cb42d6c1481d,0.0,MALE,Evening,"adobe lightroom,zomato,google home,google news...",0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,1,0,1,1,1,0,1,0,0,0,0,0,PHH,GOLD,MONTHLY,UNKNOWN,0


<div class="alert alert-block alert-info">

## Feature Engineering
</div>

### One-Hot-Encoder

In [16]:
categorical_columns = ['gender', 'ltr_segment', 'retention_segments', 'regularity_segment', 'service_affinity', 'preferredTimeBucket']

In [17]:
def one_hot_encoder(df):
    start_time = time.time()
    
    df_categorical = df[categorical_columns].copy()
    
    for column_name in categorical_columns:
        df_categorical[column_name] = df_categorical[column_name].fillna(column_name + '_UNKNOWN')
    
    # Initialize the OneHotEncoder
    encoder = OneHotEncoder()
    
    # Fit and transform the categorical columns
    encoded_categorical = encoder.fit_transform(df_categorical)
    
    # Convert the encoded result into a DataFrame
    encoded_df = pd.DataFrame(encoded_categorical.toarray(), 
                              columns=encoder.get_feature_names_out(input_features=categorical_columns))
    
    # Now drop the original categorical columns from the original DataFrame
    df = df.drop(columns=categorical_columns)

    # Concatenate the original DataFrame with the encoded DataFrame
    df = pd.concat([df, encoded_df], axis=1)

    end_time = time.time()
    execution_time = round(((end_time - start_time)*1.00/60.00), 2)
    print("Execution time:", execution_time, "mins")
    
    return df

### Vectorized Operation (Explode)

In [18]:
def vectorizer(df):
    start_time = time.time()
    
    df_dummies = df['apps_installed'].str.get_dummies(sep=',')
    
    # Convert all dummies to int32 (saves memory)
    df_dummies = df_dummies.astype('int32')

    # Use pd.concat() instead of df.join() (Faster)
    df_vectorized = pd.concat([df.drop(columns=['apps_installed']), df_dummies], axis=1)

    end_time = time.time()
    execution_time = round(((end_time - start_time)*1.00/60.00), 2)
    print("Execution time:", execution_time, "mins")
    
    return df_vectorized

### Bins definitions

In [19]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['Below-21', '21-25',  '26-30', '31-35', '36-45', 'Above-45']

In [20]:
def one_hot_encoder_to_vectorizer(df):
    print('one_hot_encoder')
    df_one_hot_encoded = one_hot_encoder(df)

    df_one_hot_encoded['age'] = df_one_hot_encoded['age'].replace(0, np.nan)

    print('vectorizer')
    df_vectorized = vectorizer(df_one_hot_encoded)

    df_vectorized['age_group'] = pd.cut(df_vectorized['age'], bins=bins, labels=labels, right=False)

    # Find columns with NaN values
    columns_with_nan = df_vectorized.columns[df_vectorized.isna().any()].tolist()
    
    # Filter DataFrame to get only columns with NaN values
    df_with_nan = df_vectorized[columns_with_nan]
    # print(df_with_nan.isnull().sum())

    return df_vectorized

<div class="alert alert-block alert-info">

## Stratified Sampling & Model
</div>

### Classification

In [21]:
def evaluate_target_vs_rest(df, target_column):
    target_classes = df[target_column].unique().tolist()
    target_classes.sort()
    classifiers = {}
    
    for target_class in target_classes:
        
        # Create a copy of df to avoid modifying the original dataframe
        df_temp = df.copy()
        
        # Split the bins target_class vs Rest 
        df_target = df_temp[df_temp[target_column] == target_class]
        df_rest = df_temp[df_temp[target_column] != target_class]
        
        df_other_sample = df_rest.sample(n=df_target.shape[0], random_state=1)
        df_combined = pd.concat([df_target, df_other_sample], ignore_index=True)
        
        # Create a binary target column where target_class is 1 and rest are 0
        df_combined['target'] = df_combined[target_column].apply(lambda x: 1 if x == target_class else 0)

        # Ensure all column names are strings
        df_combined.columns = df_combined.columns.astype(str)

        # train_test_split
        X_train, X_test, y_train, y_test = train_test_split(df_combined.drop(columns=[target_column, 'target']), 
                                                            df_combined['target'], 
                                                            test_size=0.2, 
                                                            random_state=42)

        # Train Random Forest model
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train, y_train)

        classifiers[target_class] = rf_model

        # Extract feature importances
        feature_importances = rf_model.feature_importances_

        # Get the indices of top features
        top_feature_indices = np.argsort(feature_importances)[::-1][:20] # Adjust 20 as needed
        # Get the names of top features
        top_features = X_train.columns[top_feature_indices]

        # Make predictions
        y_pred = rf_model.predict(X_test)

        # Evaluate model performance
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Output results
        print(f"Results for class {target_class} vs Rest")
        # print("Accuracy : ",  round(accuracy * 100.00, 2), "%")
        # print("Precision : ", round(precision * 100.00, 2), "%")
        # print("Recall : ", round(recall * 100.00, 2), "%")
        # print("F1 Score : ", round(f1 * 100.00, 2), "%")
        # print("Top Features:")
        
        # for feature in top_features:
            # print(feature)
        # print("---------------------------------------")
        
    return classifiers

In [22]:
def predict_age_group(df, classifiers):
    predictions = pd.DataFrame(index=df.index)
    
    for target_class, classifier in classifiers.items():
        # Ensure the features used in prediction match the training data
        X = df[[col for col in df.columns if col != 'age_group']]
        predictions[f'{target_class}'] = classifier.predict_proba(X)[:, 1]
        
    binary_predictions = (predictions > 0.5).astype(int)
    
    return binary_predictions

### Final Customer Age Class

In [23]:
# Define a function to apply the logic
def age_weight_function(row):
    cols_with_one = [col for col in ['Below-21', '21-25', '26-30', '31-35', '36-45', 'Above-45'] if row[col] == 1]
    total_sum = row['total_sum']
    
    if total_sum == 0:
        return 'UNKNOWN'
    elif total_sum == 1:
        return cols_with_one[0]
    elif total_sum == 2:
        return cols_with_one[-1]
    elif total_sum == 3 or total_sum == 5:
        return cols_with_one[len(cols_with_one) // 2]
    elif total_sum == 4:
        return cols_with_one[2]
    elif total_sum == 6:
        return cols_with_one[4]
    else:
        return 'UNKNOWN'

def get_confidence_tag(row):
    total_sum = row['total_sum']
    
    if total_sum == 0:
        return 'UNKNOWN'
    elif total_sum == 1:
        return 'VERY_HIGH'
    elif total_sum == 2:
        return 'HIGH'
    elif total_sum == 3:
        return 'MEDIUM'
    elif total_sum in [4, 5, 6]:
        return 'LOW'
    else:
        return 'UNKNOWN'

### Model 

In [24]:
def classifiers_n_predictions(df_vectorized, month, i):

    
    df_vectorized_age_w = df_vectorized[~df_vectorized['age_group'].isna()]
    df_vectorized_age_na = df_vectorized[df_vectorized['age_group'].isna()]
    
    # Keep a copy of the columns to be retained
    df_vectorized_age_na_columns = df_vectorized_age_na[['customer_id', 'age']].copy()
    
    # Drop unnecessary columns for model training
    df_vectorized_age_w.drop(columns=['customer_id', 'age'], axis=1, inplace=True)
    df_vectorized_age_na.drop(columns=['customer_id', 'age'], axis=1, inplace=True)

    
    classifiers = evaluate_target_vs_rest(df_vectorized_age_w, 'age_group')
    classifiers

    # Use the models to predict the age group for df_vectorized_age_na
    predictions = predict_age_group(df_vectorized_age_na, classifiers)
    # display(predictions.head(2))

    predictions['total_sum'] = predictions.sum(axis=1)

    # Merge the original columns with the prediction results
    df_vectorized_age_na_with_predictions = pd.concat([df_vectorized_age_na_columns['customer_id'], predictions], axis=1)
    # display(df_vectorized_age_na_with_predictions.head(2))

    print(df_vectorized_age_na_with_predictions.total_sum.value_counts(normalize=True))


    df_vectorized_age_na_with_predictions['age_group'] = df_vectorized_age_na_with_predictions.apply(age_weight_function, axis=1)
    df_vectorized_age_na_with_predictions['confidence_tag'] = df_vectorized_age_na_with_predictions.apply(get_confidence_tag, axis=1)

    
    # print(df_vectorized_age_na_with_predictions.head(2))
    print(df_vectorized_age_na_with_predictions.age_group.value_counts(normalize=True))
    print(df_vectorized_age_na_with_predictions.confidence_tag.value_counts(normalize=True))

    df_final1 = df_vectorized[~df_vectorized['age_group'].isna()][['customer_id', 'age_group']]
    df_final1['confidence_tag'] = 'VERY_HIGH'

    df_final2 = df_vectorized_age_na_with_predictions[['customer_id', 'age_group', 'confidence_tag']]

    df_predicated_age_bins = pd.concat([df_final1,df_final2])
    print(df_predicated_age_bins.shape)

    print(df_predicated_age_bins.confidence_tag.value_counts(normalize=True))

    df_predicated_age_bins['run_date'] = '2025-02-10'
    
    df_predicated_age_bins.to_csv( local_extract + 'final/customer_predicted_age_{}_{}.csv'.format(month,i), header = False, index=False)

In [26]:
for i in range(3,5):
    start_time = time.time()
    print(i)
    df_vectorized = one_hot_encoder_to_vectorizer(dfs[i])
    classifiers_n_predictions(df_vectorized, 'feb', i)
    end_time = time.time()
    execution_time = round(((end_time - start_time)*1.00/60.00), 2)
    print("Execution time:", execution_time, "mins")

3
one_hot_encoder
Execution time: 0.39 mins
vectorizer
Execution time: 9.97 mins
Results for class 21-25 vs Rest
Results for class 26-30 vs Rest
Results for class 31-35 vs Rest
Results for class 36-45 vs Rest
Results for class Above-45 vs Rest
Results for class Below-21 vs Rest
total_sum
2    0.470058
3    0.347727
1    0.094617
4    0.074274
0    0.008382
5    0.004815
6    0.000129
Name: proportion, dtype: float64
age_group
21-25       0.329447
36-45       0.283304
26-30       0.148830
31-35       0.132530
Above-45    0.079093
Below-21    0.018414
UNKNOWN     0.008382
Name: proportion, dtype: float64
confidence_tag
HIGH         0.470058
MEDIUM       0.347727
VERY_HIGH    0.094617
LOW          0.079217
UNKNOWN      0.008382
Name: proportion, dtype: float64
(1642774, 3)
confidence_tag
VERY_HIGH    0.422621
HIGH         0.299764
MEDIUM       0.221752
LOW          0.050518
UNKNOWN      0.005345
Name: proportion, dtype: float64
Execution time: 29.83 mins
4
one_hot_encoder
Execution time: 